# 01. FotMob Data Ingestion

**Stage:** Ingestion  
**Inputs:** FotMob API (season from config)  
**Outputs:** `data/raw/{season}/fixtures.parquet`, `data/raw/{season}/match_stats.parquet`, `data/raw/{season}/player_minutes.parquet`

This notebook fetches raw fixture data, match statistics, and player minutes from FotMob, performs minimal cleaning (standardizing column names and types), and saves to Parquet for downstream processing.

In [ ]:
# Load configuration
from src.config.loader import load_config

config = load_config()
season = config["data"]["seasons"][0]  # Start with first season
league_id = config["data"]["default_league_id"]

print(f"Season: {season}")
print(f"League ID: {league_id}")

In [ ]:
# NBVAL_SKIP
from src.ingestion.fotmob_client import FotMobClient

client = FotMobClient()

try:
    fixtures = client.fetch_fixtures(season=season, league_id=league_id)
    print(f"✓ Fetched {len(fixtures)} fixtures")
except Exception as e:
    print(f"✗ Error fetching fixtures: {e}")
    fixtures = None

In [ ]:
# NBVAL_SKIP
# Fetch match statistics (all fixtures)
try:
    if fixtures is not None and len(fixtures) > 0:
        match_stats_list = []
        for fixture_id in fixtures["fixture_id"].unique():
            try:
                stats = client.fetch_match_stats(fixture_id=fixture_id)
                if stats is not None and len(stats) > 0:
                    stats["fixture_id"] = fixture_id
                    match_stats_list.append(stats)
            except Exception as e:
                print(f"  Warning: Could not fetch stats for fixture {fixture_id}: {e}")
        
        if match_stats_list:
            import pandas as pd
            match_stats = pd.concat(match_stats_list, ignore_index=True)
            print(f"✓ Fetched stats for {match_stats['fixture_id'].nunique()} fixtures")
        else:
            match_stats = None
            print("✗ No match stats retrieved")
    else:
        match_stats = None
except Exception as e:
    print(f"✗ Error fetching match stats: {e}")
    match_stats = None

In [ ]:
# NBVAL_SKIP
# Fetch player minutes (all fixtures)
try:
    if fixtures is not None and len(fixtures) > 0:
        player_minutes_list = []
        for fixture_id in fixtures["fixture_id"].unique():
            try:
                minutes = client.fetch_player_minutes(fixture_id=fixture_id)
                if minutes is not None and len(minutes) > 0:
                    minutes["fixture_id"] = fixture_id
                    player_minutes_list.append(minutes)
            except Exception as e:
                print(f"  Warning: Could not fetch player minutes for fixture {fixture_id}: {e}")
        
        if player_minutes_list:
            import pandas as pd
            player_minutes = pd.concat(player_minutes_list, ignore_index=True)
            print(f"✓ Fetched player minutes for {player_minutes['fixture_id'].nunique()} fixtures")
        else:
            player_minutes = None
            print("✗ No player minutes retrieved")
    else:
        player_minutes = None
except Exception as e:
    print(f"✗ Error fetching player minutes: {e}")
    player_minutes = None

In [ ]:
import pandas as pd
from pathlib import Path

# Create output directory
output_dir = Path(f"data/raw/{season.replace('/', '-')}")
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {output_dir}")

# Save fixtures
if fixtures is not None and len(fixtures) > 0:
    fixtures_path = output_dir / "fixtures.parquet"
    fixtures.to_parquet(fixtures_path, index=False)
    print(f"✓ Saved {len(fixtures)} fixtures to {fixtures_path}")
else:
    print("✗ No fixtures to save")

# Save match statistics
if match_stats is not None and len(match_stats) > 0:
    stats_path = output_dir / "match_stats.parquet"
    match_stats.to_parquet(stats_path, index=False)
    print(f"✓ Saved {len(match_stats)} match stat rows to {stats_path}")
else:
    print("✗ No match stats to save")

# Save player minutes
if player_minutes is not None and len(player_minutes) > 0:
    minutes_path = output_dir / "player_minutes.parquet"
    player_minutes.to_parquet(minutes_path, index=False)
    print(f"✓ Saved {len(player_minutes)} player minute rows to {minutes_path}")
else:
    print("✗ No player minutes to save")

In [ ]:
# Display sample output for documentation
if fixtures is not None and len(fixtures) > 0:
    print("Sample Fixtures (first 5 rows):")
    print(fixtures.head())
    print()

if match_stats is not None and len(match_stats) > 0:
    print("Sample Match Stats (first 5 rows):")
    print(match_stats.head())
    print()

if player_minutes is not None and len(player_minutes) > 0:
    print("Sample Player Minutes (first 5 rows):")
    print(player_minutes.head())
    print()

In [ ]:
# Validate outputs
print("=== Data Validation ===")

if fixtures is not None and len(fixtures) > 0:
    print(f"✓ Fixtures: {len(fixtures)} rows, {len(fixtures.columns)} columns")
    required_fixture_cols = ["fixture_id", "home_team", "away_team", "date"]
    missing = [c for c in required_fixture_cols if c not in fixtures.columns]
    if missing:
        print(f"  ⚠ Missing columns: {missing}")
    else:
        print(f"  ✓ All required columns present")

if match_stats is not None and len(match_stats) > 0:
    print(f"✓ Match Stats: {len(match_stats)} rows, {len(match_stats.columns)} columns")
    print(f"  - {match_stats['fixture_id'].nunique()} unique fixtures")

if player_minutes is not None and len(player_minutes) > 0:
    print(f"✓ Player Minutes: {len(player_minutes)} rows, {len(player_minutes.columns)} columns")
    print(f"  - {player_minutes['fixture_id'].nunique()} unique fixtures")
    print(f"  - {player_minutes['player_id'].nunique()} unique players")

print("\n=== Ingestion Complete ===")

# FotMob Ingestion

This notebook demonstrates ingesting fixtures, match stats and player minutes.

In [ ]:
# NBVAL_SKIP
# from src.ingestion.fotmob_client import FotMobClient
# client = FotMobClient()
# fixtures = client.fetch_fixtures(season="2020/2021", league_id=1)
# fixtures.head()